In [1]:
file_path = r"C:\Users\glj523\Downloads\_FASTQ.list"

In [2]:
with open(file_path, 'r') as file:
    paths = file.read()

In [3]:
paths = paths.strip()
paths = paths.split('\n')

In [4]:
import pandas as pd

In [5]:
results = pd.DataFrame(paths, columns=['file_path'])

In [6]:
results['id'] = results['file_path'].apply(lambda x: x.split('/')[-1])

In [7]:
results['id'].apply(len).value_counts()

id
38    4912
36    1390
37     713
44     513
34     432
      ... 
69       1
84       1
83       1
67       1
55       1
Name: count, Length: 70, dtype: int64

In [8]:
results

,file_path,id
0,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-35-Ext-12-Lib-12-Index2-2JRLN-CTCTCG...
1,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-36-Ext-13-Lib-13-Index2-2JRLN-CCAAGT...
2,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-37-Ext-17-Lib-17-Index1-2JRLN-TAATAC...
3,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-38-Ext-18-Lib-18-Index1-2JRLN-ACCTTG...
4,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-39-Ext-19-Lib-19-Index1-2JRLN-ATGTAA...
...,...,...
10697,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_210pM
10698,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_210pM_rep
10699,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_260pM
10700,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_310pM


In [9]:
%env PGPASSWORD=5J8DhII0RRsPW1


env: PGPASSWORD=5J8DhII0RRsPW1


In [10]:
from pathlib import Path
import sys
import os

def find_project_root():
    path = Path(os.getcwd()).resolve()
    while path != path.root:
        if (path / 'very_rootsy_file.txt').exists():
            return path
        path = path.parent
    return None  # Project root not found

project_root = find_project_root()

sys.path.append(str(project_root))

In [11]:
from constants.db_connections import ENGINE, ENGINE_READ_ONLY, SQL_ALCH_CONFIG, PSY_CONN
q = 'select * from edna_wetlab_report'
wetlab = pd.read_sql(q, ENGINE_READ_ONLY)
q = 'select * from flowcell'
flowcell = pd.read_sql(q, ENGINE_READ_ONLY)


In [12]:
mask = results['id'].str.lower() == 'undetermined'

In [13]:
results.drop(index=results[mask].index, inplace=True)

In [14]:
mask = ~results['id'].isin(wetlab['fastq_file_id'].unique())

In [15]:
results['missing_metadata'] = mask

In [16]:
results['missing_metadata'].value_counts()

missing_metadata
False    6936
True     3617
Name: count, dtype: int64

In [17]:
capture_mask = results['id'].str.lower().str.contains('lv\d*c\d*', regex=True)

In [18]:
results['capture?'] = capture_mask

In [19]:
lv_mask = results['id'].str.lower().str.contains('lv')

In [20]:
results['contains_LV'] = lv_mask

In [21]:
results[results['contains_LV']]

,file_path,id,missing_metadata,capture?,contains_LV
89,/datasets/caeg_fastq/2022/20220718_A00706_0577...,LV7001883737-LV7000631557-CGG-3-013368,False,False,True
90,/datasets/caeg_fastq/2022/20220718_A00706_0577...,LV7001884001-LV7000631570-CGG-3-009523,False,False,True
91,/datasets/caeg_fastq/2022/20220718_A00706_0577...,LV7001884002-LV7000631554-CGG-3-009515,False,False,True
92,/datasets/caeg_fastq/2022/20220718_A00706_0577...,LV7001884003-LV7000631537-CGG-3-013379,False,False,True
93,/datasets/caeg_fastq/2022/20220718_A00706_0577...,LV7001884004-LV7000631568-CGG-3-013369,False,False,True
...,...,...,...,...,...
10697,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_210pM,True,False,True
10698,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_210pM_rep,True,False,True
10699,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_260pM,True,False,True
10700,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_310pM,True,False,True


In [22]:
control_mask = results['id'].str.lower().str.contains('ntc|control|ptc|blank')

In [23]:
results['control?'] = control_mask

In [28]:
results['missing_metadata'].value_counts()

missing_metadata
False    6936
True     3617
Name: count, dtype: int64

In [29]:
results

,file_path,id,missing_metadata,capture?,contains_LV,control?
0,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-35-Ext-12-Lib-12-Index2-2JRLN-CTCTCG...,True,False,False,False
1,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-36-Ext-13-Lib-13-Index2-2JRLN-CCAAGT...,True,False,False,False
2,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-37-Ext-17-Lib-17-Index1-2JRLN-TAATAC...,True,False,False,False
3,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-38-Ext-18-Lib-18-Index1-2JRLN-ACCTTG...,True,False,False,False
4,/datasets/caeg_fastq/2019/20191107_A00706_0052...,KapK-12-1-39-Ext-19-Lib-19-Index1-2JRLN-ATGTAA...,True,False,False,False
...,...,...,...,...,...,...
10697,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_210pM,True,False,True,False
10698,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_210pM_rep,True,False,True,False
10699,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_260pM,True,False,True,False
10700,/datasets/caeg_fastq/BaseSpace/GeoGentics_Run2...,LV7009027812-LV7008416371-LV3006007215_310pM,True,False,True,False


In [31]:
results.to_csv('fastq_metadata_check.csv', index=False)